# Tuần 2 - Chuẩn bị dữ liệu Pix3D và EDA

**Đề tài:** AI 3D Reconstruction từ ảnh 2D.

**Mục tiêu notebook:** đọc dữ liệu Pix3D, kiểm tra chất lượng dữ liệu, làm sạch cơ bản, trực quan hóa dữ liệu và chuẩn bị dữ liệu sạch cho thành viên phụ trách baseline.

## 1. Import thư viện và khai báo đường dẫn

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from PIL import Image

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RAW_DIR = PROJECT_DIR / "data" / "raw"
RESULTS_DIR = PROJECT_DIR / "results"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PIX3D_JSON = RAW_DIR / "pix3d.json"

print("PROJECT_DIR:", PROJECT_DIR)
print("PIX3D_JSON:", PIX3D_JSON)
print("Dataset tồn tại:", PIX3D_JSON.exists())

ModuleNotFoundError: No module named 'numpy'

## 2. Đọc dữ liệu Pix3D

In [ ]:
with open(PIX3D_JSON, "r", encoding="utf-8") as file:
    items = json.load(file)

df = pd.DataFrame(items)
df.head()

In [ ]:
print("Số mẫu ban đầu:", len(df))
print("Số cột:", len(df.columns))
print("Danh sách cột:", list(df.columns))

if "category" in df.columns:
    print("Số category:", df["category"].nunique())
    display(df["category"].value_counts().to_frame("Số mẫu"))

## 3. Kiểm tra lỗi dữ liệu

Mỗi mẫu Pix3D cần có đủ ảnh RGB, mask và model 3D. Nếu thiếu một trong các file này, mẫu đó không dùng được cho baseline 3D reconstruction.

In [ ]:
required_cols = ["img", "mask", "model", "category"]
missing_required_cols = [col for col in required_cols if col not in df.columns]
if missing_required_cols:
    raise ValueError(f"Thiếu cột bắt buộc trong pix3d.json: {missing_required_cols}")

df_check = df.copy()
df_check["img_exists"] = df_check["img"].apply(lambda path: (RAW_DIR / path).is_file())
df_check["mask_exists"] = df_check["mask"].apply(lambda path: (RAW_DIR / path).is_file())
df_check["model_exists"] = df_check["model"].apply(lambda path: (RAW_DIR / path).is_file())

quality_table = pd.DataFrame([
    {"Kiểm tra": "Giá trị img bị thiếu", "Số lượng": int(df_check["img"].isna().sum())},
    {"Kiểm tra": "Giá trị mask bị thiếu", "Số lượng": int(df_check["mask"].isna().sum())},
    {"Kiểm tra": "Giá trị model bị thiếu", "Số lượng": int(df_check["model"].isna().sum())},
    {"Kiểm tra": "File ảnh không tồn tại", "Số lượng": int((~df_check["img_exists"]).sum())},
    {"Kiểm tra": "File mask không tồn tại", "Số lượng": int((~df_check["mask_exists"]).sum())},
    {"Kiểm tra": "File model không tồn tại", "Số lượng": int((~df_check["model_exists"]).sum())},
    {"Kiểm tra": "Dòng trùng lặp", "Số lượng": int(df_check.duplicated(subset=required_cols).sum())},
])

quality_table

## 4. Làm sạch dữ liệu cơ bản

In [ ]:
df_clean = df_check.drop_duplicates(subset=required_cols).copy()
df_clean = df_clean.dropna(subset=required_cols)
df_clean = df_clean[
    df_clean["img_exists"]
    & df_clean["mask_exists"]
    & df_clean["model_exists"]
].copy()

df_clean.to_csv(PROCESSED_DIR / "pix3d_clean_metadata.csv", index=False, encoding="utf-8-sig")

print("Số mẫu ban đầu:", len(df))
print("Số mẫu sau làm sạch:", len(df_clean))
print("Tỷ lệ giữ lại:", round(len(df_clean) / max(len(df), 1), 4))

df_clean.head()

## 5. Bảng thống kê và biểu đồ EDA

In [ ]:
from src.utils.visualization import save_week2_visualizations

missing_counts = {
    "Thiếu ảnh": int((~df_check["img_exists"]).sum()),
    "Thiếu mask": int((~df_check["mask_exists"]).sum()),
    "Thiếu model": int((~df_check["model_exists"]).sum()),
}

image_sizes = []
for _, row in df_clean.iterrows():
    image_path = RAW_DIR / row["img"]
    with Image.open(image_path) as image:
        width, height = image.size
    image_sizes.append({"width": width, "height": height})

image_sizes_df = pd.DataFrame(image_sizes)

paths = save_week2_visualizations(
    raw_data=df,
    clean_data=df_clean,
    output_dir=RESULTS_DIR,
    category_col="category",
    missing_counts=missing_counts,
    image_sizes=image_sizes_df,
)

paths

## 6. Kiểm tra Dataset và DataLoader

Phần này chỉ kiểm tra dataloader đọc được ảnh, mask và point cloud. Baseline do thành viên khác phụ trách nên notebook này không train hoặc tính metric baseline.

In [ ]:
from src.data.dataloader import Pix3DDataset

MAX_SAMPLES = 64
NUM_POINTS = 512
IMAGE_SIZE = 224
CATEGORIES = None

dataset = Pix3DDataset(
    root_dir=RAW_DIR,
    categories=CATEGORIES,
    image_size=IMAGE_SIZE,
    num_points=NUM_POINTS,
    max_samples=MAX_SAMPLES,
)

print("Số mẫu dùng để kiểm tra dataloader:", len(dataset))
sample = dataset[0]
print("Tensor ảnh:", sample["image"].shape)
print("Point cloud:", sample["points_gt"].shape)
print("Category:", sample["category"])

## 7. Nhận xét dữ liệu ban đầu

1. Dataset Pix3D có đủ ảnh RGB, mask và model 3D, phù hợp với bài toán 3D reconstruction.
2. Bước kiểm tra missing file giúp loại bỏ các mẫu không đủ dữ liệu đầu vào hoặc ground truth.
3. Phân bố category cần được quan sát để biết dữ liệu có mất cân bằng hay không.
4. Kích thước ảnh không đồng nhất nên cần resize về cùng kích thước trước khi đưa vào mô hình.
5. File `pix3d_clean_metadata.csv` được lưu lại để thành viên phụ trách baseline có thể tái lập đúng tập dữ liệu đã làm sạch.
6. Dataloader cần đọc được ảnh, mask và point cloud ground truth trước khi chuyển sang bước baseline hoặc training.

## 8. Kế hoạch phối hợp với phần baseline

- Bàn giao `project/data/processed/pix3d_clean_metadata.csv` cho thành viên làm baseline.
- Bàn giao các biểu đồ và bảng thống kê trong `project/results/` để đưa vào báo cáo.
- Thành viên baseline sẽ dùng tập dữ liệu đã làm sạch để chạy metric phù hợp như Chamfer Distance và F-score.
- Tuần 3 nhóm sẽ so sánh kết quả baseline với mô hình chính image-to-point-cloud.
- Sau khi có checkpoint ổn định, nhóm mới chuyển sang tích hợp inference vào backend FastAPI.